# 03 · Time-aware checkpoint datasets

**DSP391m · Group 1 · FPT University** — the time-aware core of the project (RQ1).

For each t ∈ {10, 20, 40, 60, 80, 100}% we cut the clickstream and submissions at the checkpoint day (leakage rules #1 & #2), re-aggregate, and join onto the **fixed** student roster — so every checkpoint has the same 32,593 students; only the time-sliced features differ. Self-contained (no `src/` imports). Outputs: `data/checkpoints/dataset_t*.parquet`.

> Re-aggregating the 10.6M-row clickstream six times is slow, so each checkpoint is written atomically and **skipped on restart** if it already exists.

In [ ]:
import os, json, logging, warnings
from pathlib import Path
from typing import Optional
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('nb')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
INTERIM_DIR = ROOT / 'data' / 'interim'
CHECKPOINTS_DIR = ROOT / 'data' / 'checkpoints'
CHECKPOINT_MAP_PATH = ROOT / 'data' / 'checkpoint_map.csv'
REPORTS_DIR = ROOT / 'reports'
TABLES_DIR = REPORTS_DIR / 'tables'
FIGURES_DIR = REPORTS_DIR / 'figures'
CHECKPOINTS = (10, 20, 40, 60, 80, 100)
RANDOM_SEED = 42
for _d in (INTERIM_DIR, CHECKPOINTS_DIR, TABLES_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## Setup — reused helpers

The label/IO helpers and the **pure** engagement/performance aggregators are the same functions notebook 01 used at t=100%; here they run on time-sliced inputs.

In [ ]:
GROUP_COLS = ["code_module", "code_presentation", "id_student"]


PRESENTATION_KEY = ["code_module", "code_presentation"]


AT_RISK_RESULTS = ("Fail", "Withdrawn")


CANONICAL_ACTIVITY_TYPES = (
    "forumng",
    "oucontent",
    "resource",
    "homepage",
    "oucollaborate",
    "quiz",
    "subpage",
    "url",
)


def add_at_risk_label(student_info: pd.DataFrame) -> pd.DataFrame:
    """Append the binary ``at_risk`` column derived from ``final_result``."""
    out = student_info.copy()
    out["at_risk"] = out["final_result"].isin(AT_RISK_RESULTS).astype("int8")
    return out


def save_parquet_atomic(df: pd.DataFrame, path: Path) -> Path:
    """Write a DataFrame to parquet via a temp file + atomic rename.

    A crash mid-write therefore never leaves a half-written, unreadable parquet
    in place of a good one (global checkpointing rule).
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)
    return path


def load_raw_tables(raw_dir: Path = RAW_DIR) -> dict[str, pd.DataFrame]:
    """Load the small OULAD tables (everything except the 432 MB studentVle).

    studentVle is read separately by the engagement builder, which streams it in
    chunks to bound peak memory.
    """
    names = [
        "studentInfo",
        "studentRegistration",
        "studentAssessment",
        "assessments",
        "courses",
        "vle",
    ]
    return {name: pd.read_csv(raw_dir / f"{name}.csv") for name in names}

In [ ]:
_STUDENT_VLE_DTYPES = {
    "code_module": "string",
    "code_presentation": "string",
    "id_student": "int32",
    "id_site": "int32",
    "date": "int32",
    "sum_click": "int32",
}


def load_student_vle(raw_dir: Path = RAW_DIR, chunksize: int = 500_000) -> pd.DataFrame:
    """Read studentVle.csv in chunks with simple dtypes.

    Categorical conversion is done *after* the full frame is in memory; building a
    category hash table during the C parse can spike memory on large files.
    """
    path = raw_dir / "studentVle.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    log.info("Reading %s (chunksize=%d)", path.name, chunksize)
    frames = [
        c for c in pd.read_csv(path, dtype=_STUDENT_VLE_DTYPES, chunksize=chunksize)
    ]
    df = pd.concat(frames, ignore_index=True)
    log.info(
        "studentVle: %s rows, %.0f MB",
        f"{len(df):,}",
        df.memory_usage(deep=True).sum() / 1e6,
    )
    return df


def attach_activity_type(
    student_vle: pd.DataFrame, vle_meta: pd.DataFrame
) -> pd.DataFrame:
    """Map id_site -> activity_type via a lookup Series (avoids a 10M-row merge).

    id_site is globally unique in vle.csv, so the lookup is unambiguous.
    """
    if vle_meta["id_site"].duplicated().any():
        raise ValueError("id_site is not unique in vle.csv; lookup-by-site is unsafe")
    site_to_activity = vle_meta.set_index("id_site")["activity_type"]
    out = student_vle.copy()
    out["activity_type"] = out["id_site"].map(site_to_activity)
    n_missing = int(out["activity_type"].isna().sum())
    if n_missing:
        log.warning(
            "%s clicks had no activity_type (id_site absent in vle.csv)",
            f"{n_missing:,}",
        )
    return out


def aggregate_engagement(clickstream: pd.DataFrame) -> pd.DataFrame:
    """Aggregate a (full or cut) clickstream into per-student engagement features.

    ``clickstream`` must contain GROUP_COLS plus ``date``, ``sum_click`` and
    ``activity_type``. Returns one row per student-module-presentation.
    """
    required = set(GROUP_COLS) | {"date", "sum_click", "activity_type"}
    missing = required - set(clickstream.columns)
    if missing:
        raise KeyError(f"clickstream missing columns: {sorted(missing)}")

    grouped = clickstream.groupby(GROUP_COLS, observed=True)
    base = grouped.agg(
        total_clicks=("sum_click", "sum"),
        n_days_active=("date", "nunique"),
        last_active_day=("date", "max"),
    )

    # Clicks per day, then the busiest single day per student.
    daily = clickstream.groupby(GROUP_COLS + ["date"], observed=True)["sum_click"].sum()
    base["max_clicks_single_day"] = daily.groupby(level=GROUP_COLS, observed=True).max()

    # Per-type click counts, restricted to the eight canonical activity types.
    by_type = (
        clickstream.groupby(GROUP_COLS + ["activity_type"], observed=True)["sum_click"]
        .sum()
        .unstack(fill_value=0)
    )
    by_type = by_type.reindex(columns=list(CANONICAL_ACTIVITY_TYPES), fill_value=0)
    by_type.columns = [f"clicks_{c}" for c in by_type.columns]
    base = base.join(by_type)

    base["mean_clicks_per_active_day"] = (
        base["total_clicks"] / base["n_days_active"].where(base["n_days_active"] > 0)
    ).fillna(0.0)

    return base.reset_index()

In [ ]:
def aggregate_performance(
    submissions: pd.DataFrame,
    assessments: pd.DataFrame,
    cutoff_lookup: pd.DataFrame,
    roster: pd.DataFrame,
) -> pd.DataFrame:
    """Build per-student performance features as of each presentation's cutoff.

    Parameters
    ----------
    submissions   : studentAssessment (id_assessment, id_student, date_submitted,
                    is_banked, score).
    assessments   : assessments.csv (id_assessment, code_module, code_presentation,
                    assessment_type, date [deadline, NaN for the final exam], weight).
    cutoff_lookup : PRESENTATION_KEY + ``cutoff_day`` (module length for t=100%,
                    or the checkpoint day).
    roster        : every student to emit a row for (GROUP_COLS); guarantees
                    non-submitters still receive features (and not_submitted).
    """
    meta = assessments.merge(cutoff_lookup, on=PRESENTATION_KEY, how="left")
    # An assessment is "due to date" when it has a real deadline on/before cutoff.
    meta["is_due"] = meta["date"].notna() & (meta["date"] <= meta["cutoff_day"])
    due_per_pres = (
        meta[meta["is_due"]].groupby(PRESENTATION_KEY).size().rename("n_due_to_date")
    )

    sub = submissions.merge(
        meta[["id_assessment", *PRESENTATION_KEY, "weight", "cutoff_day", "is_due"]],
        on="id_assessment",
        how="left",
    )
    # Submissions counted "to date": real (not banked) and submitted by cutoff.
    submitted = sub[
        (sub["is_banked"] == 0) & (sub["date_submitted"] <= sub["cutoff_day"])
    ].copy()
    submitted["weighted"] = submitted["score"] * submitted["weight"] / 100.0

    agg = submitted.groupby(GROUP_COLS).agg(
        n_assessments_submitted=("id_assessment", "count"),
        mean_score_to_date=("score", "mean"),
        weighted_score_to_date=("weighted", "sum"),
    )
    # Of those, how many were for assessments whose deadline had passed.
    submitted_due = (
        submitted[submitted["is_due"]]
        .groupby(GROUP_COLS)
        .size()
        .rename("n_submitted_due")
    )

    out = roster[GROUP_COLS].drop_duplicates().copy()
    out = out.merge(agg, on=GROUP_COLS, how="left")
    out = out.merge(submitted_due, on=GROUP_COLS, how="left")
    out = out.merge(due_per_pres, on=PRESENTATION_KEY, how="left")

    out["n_assessments_submitted"] = (
        out["n_assessments_submitted"].fillna(0).astype("int32")
    )
    out["mean_score_to_date"] = out["mean_score_to_date"].fillna(0.0)
    out["weighted_score_to_date"] = out["weighted_score_to_date"].fillna(0.0)
    n_due = out["n_due_to_date"].fillna(0)
    n_submitted_due = out["n_submitted_due"].fillna(0)
    out["not_submitted"] = ((n_due - n_submitted_due) > 0).astype("int8")

    return out.drop(columns=["n_submitted_due", "n_due_to_date"])

## Time-aware utilities (Tasks 10–11)

`build_checkpoint_map` turns each progress % into a concrete `cutoff_day` per module-presentation (courses differ in length); `cut_at_checkpoint` keeps only rows on or before that day — exactly the information available at prediction time.

In [ ]:
def build_checkpoint_map(
    courses: pd.DataFrame, checkpoints=CHECKPOINTS
) -> pd.DataFrame:
    """Lookup table: (code_module, code_presentation, t_percent) -> cutoff_day.

    cutoff_day = round(module_presentation_length * t / 100). Long format: one row
    per module-presentation per checkpoint.
    """
    rows = []
    for _, course in courses.iterrows():
        length = course["module_presentation_length"]
        for t in checkpoints:
            rows.append(
                {
                    "code_module": course["code_module"],
                    "code_presentation": course["code_presentation"],
                    "t_percent": t,
                    "module_presentation_length": length,
                    "cutoff_day": round(length * t / 100),
                }
            )
    return pd.DataFrame(rows)


def load_checkpoint_map(path=CHECKPOINT_MAP_PATH) -> pd.DataFrame:
    return pd.read_csv(path)


def cut_at_checkpoint(
    df: pd.DataFrame,
    t_percent: int,
    checkpoint_map: pd.DataFrame,
    date_col: str = "date",
) -> pd.DataFrame:
    """Keep only rows whose ``date_col`` does not exceed the checkpoint day.

    ``df`` must contain PRESENTATION_KEY and ``date_col`` (days relative to the
    presentation start). Rows with a missing date are dropped: their timing cannot
    be verified against the checkpoint.
    """
    if t_percent not in set(checkpoint_map["t_percent"]):
        raise ValueError(
            f"t_percent={t_percent} not in checkpoint map "
            f"(available: {sorted(checkpoint_map['t_percent'].unique())})"
        )
    missing = set(PRESENTATION_KEY + [date_col]) - set(df.columns)
    if missing:
        raise KeyError(f"df is missing required columns: {sorted(missing)}")

    cutoffs = checkpoint_map.loc[
        checkpoint_map["t_percent"] == t_percent,
        PRESENTATION_KEY + ["cutoff_day"],
    ]
    merged = df.merge(cutoffs, on=PRESENTATION_KEY, how="left", validate="many_to_one")
    if merged["cutoff_day"].isna().any():
        unmatched = (
            merged.loc[merged["cutoff_day"].isna(), PRESENTATION_KEY]
            .drop_duplicates()
            .to_records(index=False)
            .tolist()
        )
        raise ValueError(
            f"module-presentations missing from checkpoint map: {unmatched}"
        )

    kept = merged[merged[date_col].notna() & (merged[date_col] <= merged["cutoff_day"])]
    return kept.drop(columns="cutoff_day").reset_index(drop=True)

## Checkpoint assembly (Task 13)

In [ ]:
DEMOGRAPHIC_COLS = [
    "gender",
    "region",
    "highest_education",
    "imd_band",
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]


def _assemble_checkpoint(
    t: int,
    clickstream: pd.DataFrame,
    raw: dict[str, pd.DataFrame],
    student_info: pd.DataFrame,
    checkpoint_map: pd.DataFrame,
) -> pd.DataFrame:
    cutoff_t = checkpoint_map.loc[
        checkpoint_map["t_percent"] == t, PRESENTATION_KEY + ["cutoff_day"]
    ]

    vle_t = cut_at_checkpoint(clickstream, t, checkpoint_map, date_col="date")
    engagement = aggregate_engagement(vle_t)
    engagement = engagement.merge(cutoff_t, on=PRESENTATION_KEY, how="left")
    engagement["days_since_last_activity"] = (
        engagement["cutoff_day"] - engagement["last_active_day"]
    ).clip(lower=0)
    engagement = engagement.drop(columns=["last_active_day", "cutoff_day"])

    performance = aggregate_performance(
        raw["studentAssessment"], raw["assessments"], cutoff_t, roster=student_info
    )

    base_cols = GROUP_COLS + DEMOGRAPHIC_COLS + ["final_result", "at_risk"]
    master_t = student_info[base_cols].copy()
    master_t = master_t.merge(
        raw["studentRegistration"][
            PRESENTATION_KEY + ["id_student", "date_registration"]
        ],
        on=GROUP_COLS,
        how="left",
        validate="many_to_one",
    )
    master_t = master_t.merge(
        engagement, on=GROUP_COLS, how="left", validate="many_to_one"
    )
    master_t = master_t.merge(
        performance, on=GROUP_COLS, how="left", validate="many_to_one"
    )
    master_t = master_t.merge(
        raw["courses"][PRESENTATION_KEY + ["module_presentation_length"]],
        on=PRESENTATION_KEY,
        how="left",
        validate="many_to_one",
    )

    # Fill engagement gaps with 0 for students absent from studentVle -- EXCEPT
    # days_since_last_activity, whose "no activity" value is NOT 0 (that would read as
    # "just active"). A student with zero activity has been idle for the whole observed
    # window, so fill it with the checkpoint's cutoff_day. At t=100 cutoff_day equals
    # module_presentation_length, so this reproduces master_raw exactly.
    engagement_fill = [
        c
        for c in engagement.columns
        if c not in GROUP_COLS and c != "days_since_last_activity"
    ]
    master_t[engagement_fill] = master_t[engagement_fill].fillna(0)
    master_t = master_t.merge(cutoff_t, on=PRESENTATION_KEY, how="left")
    master_t["days_since_last_activity"] = master_t["days_since_last_activity"].fillna(
        master_t["cutoff_day"]
    )
    master_t = master_t.drop(columns=["cutoff_day"])
    master_t.insert(0, "t_percent", t)
    return master_t

## Build the checkpoints (resumable)

In [ ]:
raw = load_raw_tables(RAW_DIR)
student_info = add_at_risk_label(raw['studentInfo'])
checkpoint_map = build_checkpoint_map(raw['courses'])
checkpoint_map.to_csv(CHECKPOINT_MAP_PATH, index=False)

todo = [t for t in CHECKPOINTS if not (CHECKPOINTS_DIR / f'dataset_t{t}.parquet').exists()]
skip = [t for t in CHECKPOINTS if t not in todo]
if skip:
    print('Resume: skipping existing checkpoints', skip)
print('Building checkpoints', todo)

clickstream = None
if todo:
    vle_parquet = INTERIM_DIR / 'studentVle.parquet'
    student_vle = pd.read_parquet(vle_parquet) if vle_parquet.exists() else load_student_vle(RAW_DIR)
    clickstream = attach_activity_type(student_vle, raw['vle'])

In [ ]:
roster_ids = set(map(tuple, student_info[GROUP_COLS].itertuples(index=False)))
stats_rows = []
for t in CHECKPOINTS:
    path = CHECKPOINTS_DIR / f'dataset_t{t}.parquet'
    if t in todo:
        df = _assemble_checkpoint(t, clickstream, raw, student_info, checkpoint_map)
        save_parquet_atomic(df, path)
        print(f'Wrote {path.name} ({len(df):,} rows x {df.shape[1]} cols)')
    else:
        df = pd.read_parquet(path)
    assert set(map(tuple, df[GROUP_COLS].itertuples(index=False))) == roster_ids, \
        f'checkpoint t={t} roster differs from the fixed student set'
    stats_rows.append({'t_percent': t, 'n_records': len(df), 'n_columns': df.shape[1],
                       'n_features': df.shape[1] - len(GROUP_COLS) - 2,
                       'at_risk_rate': round(df['at_risk'].mean(), 4)})

summary = pd.DataFrame(stats_rows)
summary.to_csv(INTERIM_DIR / 'checkpoint_summary.csv', index=False)
print(f'All six checkpoints share an identical {len(roster_ids):,}-student roster.')
display(summary)

## Conclusion

Six datasets, one fixed roster, monotonically growing information — the inputs to the time-aware discrimination analysis (notebook 02, §8) and to time-aware model training.